# SymboLM (v1.1) — Stage 2 GRPO Reinforcement Learning on Kaggle TPU VM v3-8

**Author & Principal Architect:** Sanish Gyawali  
**AI Systems Collaborator:** Antigravity (Google DeepMind)  
**Hardware:** Google Cloud TPU VM v3-8 (8 TPU cores, 128 GB HBM memory) in native `bfloat16` via PyTorch/XLA  
**Repository:** [https://github.com/gyawalisanish0/symboLM](https://github.com/gyawalisanish0/symboLM)  

---  
### Core Objectives:
1. Train policy with Group Relative Policy Optimization ($G = 4$ candidate rollouts per prompt).
2. Enforce strict SRL v1.0 AST compiler verification to permanently eliminate trailing syntax noise (`={...`, `?>"...`).
3. Reward compact self-correction mechanisms (`verify(...)` and `✗` backtracking) to raise reasoning capability.
4. Protect general intelligence via the 60/20/20 Tri-Register Curriculum.

In [ ]:
# Cell 1: Environment & Hardware Verification
import os
import sys
import torch

print('Python version:', sys.version)
print('Torch version :', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB VRAM)')
else:
    print('[Notice] No CUDA GPU found. Checking PyTorch/XLA TPU...')
    try:
        import torch_xla
        import torch_xla.core.xla_model as xm
        print('[OK] PyTorch/XLA detected! Active TPU device:', xm.xla_device())
    except Exception as e:
        print('[Notice] TPU/XLA check:', e)


In [ ]:
# Cell 2: Install required libraries
!pip install -q --upgrade pip
!pip install -q transformers peft trl accelerate sympy datasets


In [ ]:
# Cell 3: Clone latest SymboLM repository from GitHub
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM


In [ ]:
# Cell 4: Mount and unpack Stage 1 SFT Adapter from Kaggle kernel source
import os
import shutil
from pathlib import Path

target_dir = Path('/kaggle/working/symboLM/checkpoints/sft_adapter')
target_dir.mkdir(parents=True, exist_ok=True)

found = False
# 1. Search for existing extracted sft_adapter directory containing adapter_config.json
for p in Path('/kaggle/input').glob('**/sft_adapter'):
    if (p / 'adapter_config.json').exists():
        print(f'[OK] Found extracted SFT adapter directory at: {p}')
        for item in p.iterdir():
            if item.is_file():
                shutil.copy2(item, target_dir / item.name)
        found = True
        break

# 2. If not found, search for any sft*.tar.gz archive
if not found:
    for arc in Path('/kaggle/input').glob('**/*sft*.tar.gz'):
        print(f'[OK] Found SFT archive at: {arc}. Unpacking...')
        os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
        found = True
        break

print('[Status] Target SFT Adapter Directory Contents:')
for f in sorted(target_dir.iterdir()):
    sz = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name} ({sz:.2f} MB)' if f.is_file() else f'  {f.name}/')


In [ ]:
# Cell 5: Generate the 1,200-sample Tri-Register GRPO Curriculum
!python -m data.build_grpo_dataset --output ./data/grpo_curriculum.jsonl


In [ ]:
# Cell 6: Launch Stage 2 GRPO Policy Optimization
!python -m training.grpo_train \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --sft_adapter ./checkpoints/sft_adapter \
    --output_dir ./checkpoints/grpo_adapter \
    --data_dir ./data \
    --num_generations 4 \
    --learning_rate 0.000005 \
    --max_steps 250


In [ ]:
# Cell 7: Post-Training Policy Verification (Zero-Noise & Compact Self-Correction)
!python -m inference.generate \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --adapter_path ./checkpoints/grpo_adapter \
    --prompt "A train departs with 100 passengers. At the first stop, 20 leave and 35 enter. How many passengers now?"


In [ ]:
# Cell 8: Package and archive final Stage 2 policy checkpoints
!tar -czf /kaggle/working/symboLM_grpo_final_policy.tar.gz -C /kaggle/working/symboLM/checkpoints/grpo_adapter .
print("[COMPLETE] Stage 2 GRPO complete! Policy weights saved to /kaggle/working/symboLM_grpo_final_policy.tar.gz")
